In [2]:
# ============================================================================
# 00_setup_and_config.ipynb
# ----------------------------------------------------------------------------
# Beyond Full-Schema Prompting: A Graph-based Semantic Layer for Text-to-SQL
#
# Purpose:
#   Central configuration hub imported by every other notebook. Defines project
#   paths, loads the OpenAI credentials/model tiers from notebooks/.env, sets a
#   reproducible grayscale plotting style (seaborn, dpi=600, png+pdf, no caption),
#   builds a disk-cached LLM client, and exposes shared helpers.
#
#   Anonymity note: this notebook never prints filesystem paths (absolute or
#   relative), so no local directory information can leak into shared outputs.
#   All other information (settings, model tiers, metric summaries) prints normally.
#
# Run this notebook first. Other notebooks execute it via:  %run 00_setup_and_config.ipynb
# ============================================================================


# %% [markdown]
# ## Cell 1 - Install / import dependencies
# Run once per environment. Safe to skip if already installed.

# %%
# Uncomment on first run:
# !pip install openai python-dotenv pandas numpy matplotlib seaborn sqlglot tqdm networkx scipy


# %%
# Standard library
import os
import json
import hashlib
import sqlite3
import time
from pathlib import Path

# Third party
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

print("Core libraries imported.")


# %% [markdown]
# ## Cell 2 - Project paths
# Resolve the project root RELATIVE to the notebooks/ directory so the code works
# regardless of the absolute location. Expected layout:
#
#   project_root/
#     data/
#     notebooks/          <- notebooks live here (.env is uploaded here too)
#     results/
#       figures/
#       tables/
#
# Paths are NOT printed anywhere (anonymity).

# %%
# The current working directory when a notebook runs is the notebooks/ folder.
NOTEBOOKS_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOKS_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"

# Internal working directories (created if missing; not part of the required layout)
CACHE_DIR = NOTEBOOKS_DIR / ".llm_cache"        # cached LLM responses for reproducibility
CONTEXTS_DIR = RESULTS_DIR / "contexts"          # per-condition schema contexts (from notebook 01)
GENERATIONS_DIR = RESULTS_DIR / "generations"    # raw generation records (from notebook 02)

for _d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
           CACHE_DIR, CONTEXTS_DIR, GENERATIONS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print("Project directories ready.")


# %% [markdown]
# ## Cell 3 - Load environment variables (.env in notebooks/)
# The .env file is expected at notebooks/.env with keys:
#   OPENAI_API_KEY, LLM_MODEL_WEAK, LLM_MODEL_MID, LLM_MODEL_STRONG

# %%
ENV_PATH = NOTEBOOKS_DIR / ".env"
load_dotenv(dotenv_path=ENV_PATH)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model tiers act as capability-level proxies for the paper's model axis:
#   STRONG -> M-L  (large API model)
#   MID    -> mid-size tier
#   WEAK   -> small-model (SLM) proxy
# NOTE: These are capability tiers, not literal parameter counts. State this in the paper.
MODEL_WEAK = os.getenv("LLM_MODEL_WEAK")
MODEL_MID = os.getenv("LLM_MODEL_MID")
MODEL_STRONG = os.getenv("LLM_MODEL_STRONG")

# Ordered from weakest to strongest; used by RQ4 interaction analysis.
MODEL_TIERS = {
    "WEAK": MODEL_WEAK,
    "MID": MODEL_MID,
    "STRONG": MODEL_STRONG,
}
# Human-readable labels for figures (kept model-agnostic).
MODEL_TIER_LABELS = {"WEAK": "Weak", "MID": "Mid", "STRONG": "Strong"}
MODEL_TIER_ORDER = ["WEAK", "MID", "STRONG"]

assert OPENAI_API_KEY, "OPENAI_API_KEY not found. Place .env in the notebooks/ folder."
assert all(MODEL_TIERS.values()), f"Missing model name(s) in .env: {MODEL_TIERS}"

print("Environment loaded. Model tiers:")
for k in MODEL_TIER_ORDER:
    print(f"  {k:6s} -> {MODEL_TIERS[k]}")


# %% [markdown]
# ## Cell 4 - Experiment axes (mirror the planning document)
# Conditions C1..C5, dataset ids, and metric names are declared here so every
# notebook references the same canonical strings.

# %%
# Retrieval / context conditions
CONDITIONS = {
    "C1": "Full Schema",
    "C2": "Vector-only",
    "C3": "Graph-only",
    "C4": "Hybrid",
    "C5": "Hybrid+Pruning",
}
CONDITION_ORDER = ["C1", "C2", "C3", "C4", "C5"]

# Datasets
DATASETS = {
    "D1": "Spider",
    "D2": "BIRD",
    "D2_fin": "BIRD-finance-proxy",  # domain-glossary-heavy slice for RQ5
    "D3": "In-house-finance",        # structure private; aggregate metrics only
}

# Random seed for any sampling / shuffling
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

print("Conditions:", CONDITION_ORDER)
print("Datasets   :", list(DATASETS.keys()))


# %% [markdown]
# ## Cell 5 - Grayscale plotting style
# Requirements: seaborn grayscale, no caption baked into the image, dpi=600,
# save BOTH .png and .pdf. Because grayscale removes color as a channel, we also
# vary line style, marker, and hatch so series stay distinguishable in B/W print.

# %%
sns.set_theme(style="white", context="paper")

# Global rcParams for publication-grade grayscale figures.
mpl.rcParams.update({
    "figure.dpi": 120,          # on-screen preview; saved figures override to 600
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "0.85",
    "grid.linewidth": 0.5,
    "legend.frameon": False,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "image.cmap": "Greys",
    "figure.autolayout": True,
})

# Discrete grayscale palette (dark -> light) for categorical series.
def gray_palette(n):
    """Return n evenly spaced grays from dark (0.15) to light (0.75)."""
    if n <= 1:
        return ["0.2"]
    return [str(round(v, 3)) for v in np.linspace(0.15, 0.75, n)]

# Redundant encodings so grayscale series remain distinguishable in print.
LINE_STYLES = ["-", "--", "-.", ":", (0, (3, 1, 1, 1))]
MARKERS = ["o", "s", "^", "D", "v", "P", "X"]
HATCHES = ["", "///", "...", "xxx", "\\\\\\", "ooo"]

# Fixed grayscale + style mapping per condition so the same condition looks the
# same across every figure in the paper.
CONDITION_STYLE = {}
_grays = gray_palette(len(CONDITION_ORDER))
for _i, _c in enumerate(CONDITION_ORDER):
    CONDITION_STYLE[_c] = {
        "color": _grays[_i],
        "linestyle": LINE_STYLES[_i % len(LINE_STYLES)],
        "marker": MARKERS[_i % len(MARKERS)],
        "hatch": HATCHES[_i % len(HATCHES)],
        "label": f"{_c} ({CONDITIONS[_c]})",
    }

print("Grayscale plotting style configured.")
print("Condition styles:", {k: v["color"] for k, v in CONDITION_STYLE.items()})


# %% [markdown]
# ## Cell 6 - Figure and table saving helpers
# Save figures to results/figures/ as BOTH png and pdf at dpi=600, no caption
# drawn on the canvas. Save tables to results/tables/ as CSV + LaTeX. Filenames
# are passed WITHOUT extension.
#
# Anonymity: these helpers confirm the save WITHOUT printing any path; they print
# only the base filename so no directory information is exposed.

# %%
def save_figure(fig, name, subdir=None):
    """
    Save a figure as {name}.png and {name}.pdf under results/figures/[subdir/].
    - dpi is enforced at 600 via rcParams savefig.dpi.
    - No caption is added to the image (per requirement).
    - Prints only the base filename (no path). Returns the list of written paths.
    """
    out_dir = FIGURES_DIR if subdir is None else (FIGURES_DIR / subdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    written = []
    for ext in ("png", "pdf"):
        path = out_dir / f"{name}.{ext}"
        fig.savefig(path, format=ext)  # dpi/bbox come from rcParams
        written.append(path)
    print(f"Saved figure: {name}.png, {name}.pdf")
    return written


def save_table(df, name, subdir=None, index=False, float_format="%.4f"):
    """
    Save a DataFrame to results/tables/ as CSV (machine-readable) and a LaTeX
    booktabs snippet (paper-ready). Filenames passed WITHOUT extension.
    Prints only the base filename (no path). Returns the CSV path.
    """
    out_dir = TABLES_DIR if subdir is None else (TABLES_DIR / subdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / f"{name}.csv"
    tex_path = out_dir / f"{name}.tex"
    df.to_csv(csv_path, index=index)
    try:
        df.to_latex(tex_path, index=index, float_format=float_format,
                    escape=True, bold_rows=False)
    except Exception:
        pass
    print(f"Saved table: {name}.csv, {name}.tex")
    return csv_path


# %% [markdown]
# ## Cell 7 - Disk-cached OpenAI client
# Every LLM call is keyed by a hash of (model, messages, params). Identical calls
# are served from disk (reproducibility + no duplicate spend). Delete .llm_cache/
# to force fresh calls.
#
# Token-limit parameter differs by model generation: newer models (gpt-5.x)
# require 'max_completion_tokens' and reject 'max_tokens'; older models
# (gpt-4o-mini) use 'max_tokens'. We try the new name first and fall back once,
# caching which name a model accepts so we don't pay the failed call twice.

# %%
from openai import OpenAI

_client = OpenAI(api_key=OPENAI_API_KEY, timeout=90.0, max_retries=0)

# Remembers, per model, which token-limit kwarg the API accepts.
_TOKEN_PARAM_CACHE = {}


def _cache_key(model, messages, extra):
    payload = json.dumps(
        {"model": model, "messages": messages, "extra": extra},
        sort_keys=True, ensure_ascii=False,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _create_completion(model, messages, temperature, max_tokens):
    """
    Call chat.completions handling the max_tokens vs max_completion_tokens split.
    Also handles newer models that only allow the default temperature.
    """
    # Choose the token param: use the remembered one, else try the new name first.
    token_param = _TOKEN_PARAM_CACHE.get(model, "max_completion_tokens")

    def _attempt(tp, temp):
        kwargs = {"model": model, "messages": messages, tp: max_tokens}
        if temp is not None:
            kwargs["temperature"] = temp
        return _client.chat.completions.create(**kwargs)

    for temp_try in (temperature, None):     # fall back to default temperature
        for tp_try in (token_param, "max_tokens", "max_completion_tokens"):
            try:
                resp = _attempt(tp_try, temp_try)
                _TOKEN_PARAM_CACHE[model] = tp_try   # remember what worked
                return resp
            except Exception as e:
                msg = str(e).lower()
                # If the token-param name is the problem, try the other name.
                if "max_tokens" in msg or "max_completion_tokens" in msg:
                    continue
                # If temperature is the problem, break to retry without it.
                if "temperature" in msg:
                    break
                # Any other error: re-raise (network/rate-limit handled by caller).
                raise
    # Last resort: minimal call with the new token param and no temperature.
    resp = _attempt("max_completion_tokens", None)
    _TOKEN_PARAM_CACHE[model] = "max_completion_tokens"
    return resp


def llm_complete(prompt, model=None, system=None, temperature=0.0,
                 max_tokens=1024, use_cache=True, max_retries=4):
    """
    Single-turn completion with disk caching and exponential-backoff retry.
    Returns: {text, model, prompt_tokens, completion_tokens, total_tokens,
              cached, latency_s}.
    """
    model = model or MODEL_STRONG
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    extra = {"temperature": temperature, "max_tokens": max_tokens}
    key = _cache_key(model, messages, extra)
    cache_path = CACHE_DIR / f"{key}.json"

    if use_cache and cache_path.exists():
        rec = json.loads(cache_path.read_text(encoding="utf-8"))
        rec["cached"] = True
        rec["latency_s"] = 0.0
        return rec

    last_err = None
    for attempt in range(max_retries):
        try:
            t0 = time.time()
            resp = _create_completion(model, messages, temperature, max_tokens)
            latency = time.time() - t0
            usage = resp.usage
            rec = {
                "text": resp.choices[0].message.content or "",
                "model": model,
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
                "cached": False,
                "latency_s": round(latency, 3),
            }
            cache_path.write_text(json.dumps(rec, ensure_ascii=False), encoding="utf-8")
            return rec
        except Exception as e:
            last_err = e
            wait = 2.0 * (2 ** attempt)
            print(f"  LLM call failed (attempt {attempt+1}/{max_retries}): {e} "
                  f"-> retry in {wait:.0f}s")
            time.sleep(wait)

    raise RuntimeError(f"LLM call failed after {max_retries} retries: {last_err}")


# %% [markdown]
# ## Cell 8 - Pricing table for cost analysis (RQ2 / Section 10)
# $ per 1M tokens. EDIT these to match current pricing for your models before
# running the cost notebook. Kept here so cost math is centralized.

# %%
# Format: model_name -> {"in": $/1M input tokens, "out": $/1M output tokens}
# Fill in the real numbers for your billing; placeholders below are examples.
PRICING_USD_PER_1M = {
    MODEL_WEAK:   {"in": 0.15, "out": 0.60},
    MODEL_MID:    {"in": 0.50, "out": 2.00},
    MODEL_STRONG: {"in": 2.50, "out": 10.00},
}


def estimate_cost(model, prompt_tokens, completion_tokens):
    """Return USD cost for a single call given token counts."""
    p = PRICING_USD_PER_1M.get(model)
    if p is None or prompt_tokens is None or completion_tokens is None:
        return np.nan
    return (prompt_tokens / 1e6) * p["in"] + (completion_tokens / 1e6) * p["out"]


# %% [markdown]
# ## Cell 9 - Sanity check (optional; makes one tiny live call)
# Set RUN_LIVE_CHECK = True to verify credentials end-to-end. Costs a few tokens.

# %%
RUN_LIVE_CHECK = False

if RUN_LIVE_CHECK:
    r = llm_complete("Reply with the single word: OK",
                     model=MODEL_WEAK, max_tokens=5)
    print("Live check response:", repr(r["text"]))
    print("Tokens:", r["total_tokens"], "| cached:", r["cached"])
else:
    print("Live check skipped (set RUN_LIVE_CHECK=True to test credentials).")


# %% [markdown]
# ## Cell 10 - Persist a non-secret configuration snapshot
# Written to disk for auditability. No path is printed.

# %%
_config_snapshot = {
    "conditions": CONDITIONS,
    "datasets": DATASETS,
    "model_tiers": {k: MODEL_TIERS[k] for k in MODEL_TIER_ORDER},
    "seed": GLOBAL_SEED,
    "savefig_dpi": mpl.rcParams["savefig.dpi"],
}
(RESULTS_DIR / "config_snapshot.json").write_text(
    json.dumps(_config_snapshot, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Setup complete. Config snapshot saved.")

Core libraries imported.
Project directories ready.
Environment loaded. Model tiers:
  WEAK   -> gpt-4o-mini
  MID    -> gpt-5.4-mini
  STRONG -> gpt-5.4
Conditions: ['C1', 'C2', 'C3', 'C4', 'C5']
Datasets   : ['D1', 'D2', 'D2_fin', 'D3']
Grayscale plotting style configured.
Condition styles: {'C1': '0.15', 'C2': '0.3', 'C3': '0.45', 'C4': '0.6', 'C5': '0.75'}
Live check skipped (set RUN_LIVE_CHECK=True to test credentials).
Setup complete. Config snapshot saved.
